<a href="https://colab.research.google.com/github/Shivamthakur7500/FireWatchNet/blob/main/09_Sentinel2_Indices.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
ROOT = "/content/drive/MyDrive/FireWatchNet"

Mounted at /content/drive


In [ ]:
import os, glob
ROOT = "/content/drive/MyDrive/FireWatchNet"
exts = (".geojson", ".gpkg", ".shp", ".parquet", ".csv")
for p in sorted(glob.glob(f"{ROOT}/**/*", recursive=True)):
    if p.lower().endswith(exts) and any(k in p.lower() for k in ("grid", "cell")):
        print(p)

/content/drive/MyDrive/FireWatchNet/Data/processed/Satellite/Satellite_Patch_to_Grid_Mapping.csv
/content/drive/MyDrive/FireWatchNet/Data/processed/Satellite/Satellite_Patch_to_Grid_Mapping_Filtered.csv
/content/drive/MyDrive/FireWatchNet/Data/processed/Satellite/ViT_Features/ViT_Features_By_Grid_Cell.csv
/content/drive/MyDrive/FireWatchNet/Data/processed/Spatial/Uttarakhand_5km_Grid_UTM.geojson
/content/drive/MyDrive/FireWatchNet/Data/processed/Spatial/Uttarakhand_5km_Grid_WGS84.geojson
/content/drive/MyDrive/FireWatchNet/Data/processed/Spatial/VIIRS_Cell_Detection_Summary_5km_2023_2024.csv
/content/drive/MyDrive/FireWatchNet/Data/processed/Spatial/VIIRS_Grid_7Day_Forecast_Labels_5km_2023_2024.csv


In [ ]:
# Colab, run in order. Needs a free Earth Engine account/project.
import ee, json, pandas as pd, geopandas as gpd
ee.Authenticate()
ee.Initialize(project="firewatchnet")

ROOT = "/content/drive/MyDrive/FireWatchNet"
grid = gpd.read_file(f"{ROOT}/Data/processed/Spatial/Uttarakhand_5km_Grid_WGS84.geojson")
grid = grid.to_crs(4326)[["cell_id", "geometry"]]    # use your real cell id column
fc = ee.FeatureCollection(json.loads(grid.to_json()))

CS = ee.ImageCollection("GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED")

def monthly(year, month):
    start = ee.Date.fromYMD(year, month, 1)
    end = start.advance(1, "month")
    col = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
           .filterDate(start, end).filterBounds(fc)
           .linkCollection(CS, ["cs_cdf"])
           .map(lambda im: im.updateMask(im.select("cs_cdf").gte(0.6))))
    im = col.median().divide(10000)
    nir, red, swir1, swir2 = im.select("B8"), im.select("B4"), im.select("B11"), im.select("B12")
    out = ee.Image.cat([
        nir.subtract(red).divide(nir.add(red)).rename("ndvi"),
        nir.subtract(swir2).divide(nir.add(swir2)).rename("nbr"),
        nir.subtract(swir1).divide(nir.add(swir1)).rename("ndmi"),
        im.select("B8").mask().rename("valid")])  # 1 where a clear pixel exists
    stats = out.reduceRegions(fc, ee.Reducer.mean(), scale=100, tileScale=4)
    return stats.map(lambda f: f.set({"year": year, "month": month, "n_images": col.size()}))

rows = []
for y, m in [(y, m) for y in (2022, 2023, 2024) for m in range(1, 13)
             if (y, m) >= (2022, 10) and (y, m) <= (2024, 11)]:
    data = monthly(y, m).getInfo()["features"]
    rows += [dict(f["properties"]) for f in data]
    print(y, m, len(data))

df = pd.DataFrame(rows)
df.to_parquet(f"{ROOT}/Data/processed/s2_monthly_indices.parquet")

2022 10 2233
2022 11 2233
2022 12 2233
2023 1 2233
2023 2 2233
2023 3 2233
2023 4 2233
2023 5 2233
2023 6 2233
2023 7 2233
2023 8 2233
2023 9 2233
2023 10 2233
2023 11 2233
2023 12 2233
2024 1 2233
2024 2 2233
2024 3 2233
2024 4 2233
2024 5 2233
2024 6 2233
2024 7 2233
2024 8 2233
2024 9 2233
2024 10 2233
2024 11 2233


In [ ]:
import pandas as pd
ROOT = "/content/drive/MyDrive/FireWatchNet"
s2 = pd.read_parquet(f"{ROOT}/Data/processed/s2_monthly_indices.parquet")

print(s2.shape)                       # expect (58058, ~8)
print(s2.columns.tolist())
print(s2["cell_id"].nunique())        # expect 2233
print(s2[["ndvi", "nbr", "ndmi", "valid"]].describe().round(3))

# missing values and clear-pixel coverage by month
chk = s2.groupby(["year", "month"]).agg(
    ndvi_missing=("ndvi", lambda x: x.isna().mean().round(3)),
    valid_mean=("valid", "mean"))
print(chk.round(3))

(58058, 8)
['cell_id', 'month', 'n_images', 'nbr', 'ndmi', 'ndvi', 'valid', 'year']
2233
            ndvi        nbr       ndmi      valid
count  57800.000  57800.000  57800.000  58058.000
mean       0.478      0.399      0.209      0.949
std        0.260      0.175      0.196      0.167
min       -0.266     -0.309     -0.319      0.000
25%        0.349      0.274      0.081      1.000
50%        0.550      0.410      0.181      1.000
75%        0.682      0.525      0.267      1.000
max        0.959      0.919      0.917      1.000
            ndvi_missing  valid_mean
year month                          
2022 10            0.000       1.000
     11            0.000       1.000
     12            0.000       1.000
2023 1             0.000       1.000
     2             0.000       1.000
     3             0.000       1.000
     4             0.000       0.999
     5             0.000       0.999
     6             0.003       0.921
     7             0.001       0.797
     8           

In [ ]:
import pandas as pd
ROOT = "/content/drive/MyDrive/FireWatchNet"
ds = pd.read_parquet(f"{ROOT}/Data/processed/firewatchnet_dataset.parquet")
print(ds.columns.tolist())
print(ds.index.names)
print(ds.head(3))

['cell_id', 'forecast_date', 'fire_next_7d', 'prior_7d_days', 'prior_30d_days', 'had_detection_prior_7d', 'had_detection_prior_30d', 'x_utm', 'y_utm', 'lat', 'lon', 'doy_sin', 'doy_cos', 'month', 'temperature_2m_c', 'precipitation_mm', 'volumetric_soil_water_layer_1', 'wind_speed', 'precip_7d_mm', 'precip_30d_mm', 'temp_7d_c', 'soil_7d', 'split', 'cum_days_before']
[None]
             cell_id forecast_date  fire_next_7d  prior_7d_days  \
0  UK_170000_3365000    2023-01-01             0            0.0   
1  UK_170000_3365000    2023-01-02             0            0.0   
2  UK_170000_3365000    2023-01-03             0            0.0   

   prior_30d_days  had_detection_prior_7d  had_detection_prior_30d  \
0             0.0                       0                        0   
1             0.0                       0                        0   
2             0.0                       0                        0   

           x_utm         y_utm        lat  ...  temperature_2m_c  \
0  1728

In [ ]:
import pandas as pd
ROOT = "/content/drive/MyDrive/FireWatchNet"

s2 = pd.read_parquet(f"{ROOT}/Data/processed/s2_monthly_indices.parquet")
ds = pd.read_parquet(f"{ROOT}/Data/processed/firewatchnet_dataset.parquet")

DATE_COL = "forecast_date"
ID_COL = "cell_id"

# same type for the cell ids in both tables
ds[ID_COL] = ds[ID_COL].astype(str)
s2["cell_id"] = s2["cell_id"].astype(str)
print("cells in common:", len(set(ds[ID_COL]) & set(s2["cell_id"])), "(expect 2233)")

# composite month, and the first day it may be used (the next month)
s2["s2_month"] = pd.to_datetime(s2[["year", "month"]].assign(day=1)).astype("datetime64[ns]")
s2["join_month"] = (s2["s2_month"] + pd.offsets.MonthBegin(1)).astype("datetime64[ns]")
s2 = s2.rename(columns={"ndvi": "s2_ndvi", "nbr": "s2_nbr",
                        "ndmi": "s2_ndmi", "valid": "s2_valid"})
s2 = s2[[ID_COL, "join_month", "s2_month", "s2_ndvi", "s2_nbr", "s2_ndmi", "s2_valid"]]

ds[DATE_COL] = pd.to_datetime(ds[DATE_COL])
ds["join_month"] = ds[DATE_COL].dt.to_period("M").dt.to_timestamp().astype("datetime64[ns]")

n0 = len(ds)
ds = ds.merge(s2, on=[ID_COL, "join_month"], how="left")
assert len(ds) == n0, "join changed the row count"

# leakage check: every composite must come from a month BEFORE the row's month
has = ds["s2_month"].notna()
assert (ds.loc[has, "s2_month"] < ds.loc[has, "join_month"]).all()
print("leakage check passed")

print(ds[["s2_ndvi", "s2_nbr", "s2_ndmi"]].isna().mean().round(4))
ds = ds.drop(columns=["join_month"])
ds.to_parquet(f"{ROOT}/Data/processed/firewatchnet_dataset_s2.parquet")
print(ds.shape)

cells in common: 2233 (expect 2233)
leakage check passed
s2_ndvi    0.005
s2_nbr     0.005
s2_ndmi    0.005
dtype: float64
(1585430, 29)


In [ ]:
import pandas as pd, numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score, roc_auc_score

ROOT = "/content/drive/MyDrive/FireWatchNet"
ds = pd.read_parquet(f"{ROOT}/Data/processed/firewatchnet_dataset_s2.parquet")

TARGET = "fire_next_7d"
S2 = ["s2_ndvi", "s2_nbr", "s2_ndmi", "s2_valid"]
EXCLUDE = ["cell_id", "forecast_date", TARGET, "split", "s2_month"] + S2
BASE = [c for c in ds.columns if c not in EXCLUDE]
print(len(BASE), "base features:", BASE)

tr, va, te = (ds[ds.split == s] for s in ("train", "val", "test"))
evals = {"E1": (tr, va), "E2": (pd.concat([tr, va]), te)}

def run(train, test, feats, seed):
    # note: this uses the median of the training rows to fill missing values
    med = train[feats].median()
    s = train.sample(300_000, random_state=seed)
    rf = RandomForestClassifier(n_estimators=100, max_depth=12, class_weight="balanced",
                                n_jobs=-1, random_state=seed)
    rf.fit(s[feats].fillna(med), s[TARGET])
    p = rf.predict_proba(test[feats].fillna(med))[:, 1]
    return average_precision_score(test[TARGET], p), roc_auc_score(test[TARGET], p), rf

rows = []
for name, (train, test) in evals.items():
    for label, feats in {"baseline": BASE,
                         "+S2 indices": BASE + ["s2_ndvi", "s2_nbr", "s2_ndmi", "s2_valid"]}.items():
        for seed in (0, 1, 2):
            pr, roc, rf = run(train, test, feats, seed)
            rows.append(dict(eval=name, model=label, seed=seed, pr_auc=pr, roc_auc=roc))
            print(name, label, seed, round(pr, 4))

res = pd.DataFrame(rows)
summary = res.groupby(["eval", "model"])[["pr_auc", "roc_auc"]].agg(["mean", "std"]).round(4)
print(summary)
res.to_csv(f"{ROOT}/Results/s2_ablation.csv", index=False)

20 base features: ['prior_7d_days', 'prior_30d_days', 'had_detection_prior_7d', 'had_detection_prior_30d', 'x_utm', 'y_utm', 'lat', 'lon', 'doy_sin', 'doy_cos', 'month', 'temperature_2m_c', 'precipitation_mm', 'volumetric_soil_water_layer_1', 'wind_speed', 'precip_7d_mm', 'precip_30d_mm', 'temp_7d_c', 'soil_7d', 'cum_days_before']
E1 baseline 0 0.241
E1 baseline 1 0.2189
E1 baseline 2 0.2496
E1 +S2 indices 0 0.2724
E1 +S2 indices 1 0.268
E1 +S2 indices 2 0.2704
E2 baseline 0 0.2184
E2 baseline 1 0.2009
E2 baseline 2 0.1864
E2 +S2 indices 0 0.1865
E2 +S2 indices 1 0.1516
E2 +S2 indices 2 0.1367
                  pr_auc         roc_auc        
                    mean     std    mean     std
eval model                                      
E1   +S2 indices  0.2703  0.0022  0.7674  0.0035
     baseline     0.2365  0.0158  0.7495  0.0027
E2   +S2 indices  0.1583  0.0255  0.9460  0.0022
     baseline     0.2019  0.0160  0.9449  0.0018
